<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install the third-party libraries used to pull option chains and plot the volatility surface so the notebook runs end-to-end on a fresh environment.

In [ ]:
!pip install yfinance pandas numpy matplotlib

If you are running in a locked-down environment (some corporate JupyterHub setups), you may need to ask for yfinance to be whitelisted or installed via your platform’s package manager instead of pip.

## Imports and setup

We use datetime for “days to expiration,” pandas for reshaping the option chain into a grid, numpy for meshing coordinates for 3D plotting, matplotlib for charts, and yfinance to fetch live option chains from Yahoo Finance.

In [ ]:
import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

Pull every listed expiration for a ticker and standardize the result into one DataFrame so we can later compare strikes and maturities as a single surface instead of isolated rows.

In [ ]:
def option_chains(ticker):
    asset = yf.Ticker(ticker)
    expirations = asset.options

    chains = pd.DataFrame()

    for expiration in expirations:
        opt = asset.option_chain(expiration)

        calls = opt.calls
        calls["optionType"] = "call"

        puts = opt.puts
        puts["optionType"] = "put"

        chain = pd.concat([calls, puts])
        chain["expiration"] = pd.to_datetime(expiration) + pd.DateOffset(
            hours=23,
            minutes=59,
            seconds=59,
        )

        chains = pd.concat([chains, chain])

    chains["daysToExpiration"] = (
        chains.expiration - dt.datetime.today()
    ).dt.days + 1

    return chains

This “one table” shape is the bridge from raw quotes to analysis: once the chain is consolidated, we can slice skew, slice term structure, and then reshape into a strike-by-maturity grid. Adding a consistent expiration timestamp also avoids subtle grouping bugs when expirations come in as strings.

## Pull options chain across expirations

Fetch SPY’s full option chain and isolate calls so our first surface focuses on one option type with a cleaner skew signal.

In [ ]:
options = option_chains("SPY")
calls = options[options["optionType"] == "call"]

Working with a liquid ETF like "SPY" helps because you typically get enough strikes and expirations to see real structure rather than sparse, jumpy points. Splitting calls and puts early is also practical because the skew often differs by type due to supply/demand and moneyness conventions.

Quickly list expirations available in the downloaded data so we know what maturities we can build the grid from.

In [ ]:
set(calls.expiration)

This check is a small but useful workflow habit: when beginners think “the data is wrong,” it’s often just that the expiry set changed or a filter removed most rows. Seeing the available expirations keeps our later slices and plots grounded in what we actually pulled.

## Slice skew and term structure

Take one expiration to study skew across strikes, and take one strike to study the term structure across expirations, filtering out near-zero implied vols that are usually stale or broken quotes.

In [ ]:
calls_at_expiry = calls[calls["expiration"] == "2026-08-21 23:59:59"]
filtered_calls_at_expiry = calls_at_expiry[
    calls_at_expiry.impliedVolatility >= 0.001
]

In [ ]:
calls_at_strike = calls[calls["strike"] == 680.0]
filtered_calls_at_strike = calls_at_strike[
    calls_at_strike.impliedVolatility >= 0.001
]

These two “1D cuts” are the simplest way to stop thinking in single-vol terms: skew tells us how vol changes with strike, and term structure tells us how vol changes with maturity. The small impliedVolatility filter is not cosmetic; it prevents a few bad points from dominating the chart and misleading our interpretation.

Reshape call implied vols into a strike (rows) by days-to-expiration (columns) grid, which is the core object we need for a clean surface plot.

In [ ]:
surface = (
    calls[["daysToExpiration", "strike", "impliedVolatility"]]
    .pivot_table(
        values="impliedVolatility",
        index="strike",
        columns="daysToExpiration",
    )
    .dropna()
)

Pivoting is the “professional workflow” step in this post: it turns a long options table into a coherent surface you can sanity check, model, and compare over time. Dropping NaNs is a pragmatic way to get a rectangular grid for plotting, but it also teaches an important lesson that real option chains are incomplete, so filtering choices shape what you see.

Plot the skew at a fixed expiration and the term structure at a fixed strike so we can visually confirm structure before moving to 3D.

In [ ]:
filtered_calls_at_expiry[["strike", "impliedVolatility"]].set_index(
    "strike"
).plot(
    title="Implied Volatility Skew",
    figsize=(7, 4),
)

In [ ]:
filtered_calls_at_strike[["expiration", "impliedVolatility"]].set_index(
    "expiration"
).plot(
    title="Implied Volatility Term Structure",
    figsize=(7, 4),
)

These plots act like unit tests for our surface: if skew and term structure look unreasonable, it’s better to debug filters and data alignment here than after a 3D plot hides the issue. In practice, desks will check these slices routinely because they map directly to common risk narratives like “skew steepened” or “front vol jumped.”

## Plot the full implied vol surface

Build a 3D surface plot from the grid so we can see skew and term structure together as one object rather than flipping between expirations by hand.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
x, y, z = surface.columns.values, surface.index.values, surface.values
X, Y = np.meshgrid(x, y)
ax.set_xlabel("Days to expiration")
ax.set_ylabel("Strike price")
ax.set_zlabel("Implied volatility")
ax.set_title("Call implied volatility surface")
ax.plot_surface(X, Y, z)
plt.show()

The surface is the payoff: it makes market structure obvious and helps us avoid false “bad data” hunts when the shape is actually real pricing information. Once we have this, we can start asking better questions, like which region of the surface moved after an event or whether our hedges are sensitive to skew versus overall vol level.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.